# Bronze Ingestion

The logic lives in `src/bronze.py` and `src/common.py`. This notebook only
wires things up and runs it, so the same code can be run by a Job later.

Per table: `find_watermark -> get_latest_wm (ceiling) -> pull_delta -> push_delta -> update_wm`

A table that fails is recorded and skipped. The rest still run.


## 1. Make `src/` importable

`src/` sits next to `notebooks/` in the repo, so we step up one level and add it
to the path. Without this, `import bronze` cannot find the file.


In [ ]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
SRC = os.path.join(REPO_ROOT, "src")

if SRC not in sys.path:
    sys.path.insert(0, SRC)

print("repo:", REPO_ROOT)
print("src on path:", SRC in sys.path)

## 2. Connect to the source

`connect()` builds the JDBC url, user and password into one dict. The password
comes out of the Databricks secret scope - it is never typed into the notebook.


In [ ]:
from common import connect

conn = connect(
    server="klitikalwa.database.windows.net",
    database="free-sql-db-9534149",
    user="adminzapas",
    scope="ap_source",
    key="azsql_password",
    dbutils=dbutils,
)

print("connected as", conn["user"])

## 3. Config

Two steps, and it matters which one the pipeline reads:

1. `load_config` + `process_config` read the YAML **file** in the repo
2. `sync_config_to_control` writes that into **`workspace.control.tables`**
3. `get_enabled_table_configs` reads it back out - and *this* is what the
   pipeline runs on

Why the round trip? In production there is no repo checked out next to the job,
so the YAML disappears and `control.tables` becomes the source of truth. Reading
from the table here means dev and prod run the exact same code path.


In [ ]:
from common import load_config, process_config, sync_config_to_control, get_enabled_table_configs

CONFIG_PATH = os.path.join(REPO_ROOT, "config", "ingestion_config.yaml")

file_configs = process_config(load_config(CONFIG_PATH))
synced = sync_config_to_control(spark, file_configs)
print(f"synced {synced} table configs into control.tables")

configs = get_enabled_table_configs(spark)
print(f"{len(configs)} enabled tables\n")

for cfg in configs:
    print(f"  {cfg['source_table']:26s} {cfg['strategy']:12s} -> {cfg['target_table']}")

## 4. Run

`run_bronze` returns a summary dict instead of raising, so one bad table does
not turn the whole job red. Check `result["failed"]` to see what went wrong.


In [ ]:
from bronze import run_bronze

result = run_bronze(spark, conn, configs)

print("\nrun_id:", result["run_id"])
print("failed:", result["failed"])

## 5. Verify

In [ ]:
%sql
SELECT table_name, watermark_column, last_watermark, last_successful_run_id, updated_at
FROM   workspace.control.watermarks
ORDER  BY table_name

In [ ]:
%sql
SELECT run_id, table_name, status, started_at, finished_at, error_message
FROM   workspace.control.pipeline_runs
ORDER  BY started_at DESC
LIMIT  40

### Did it actually land?

Bronze is a mirror of the source, so the row counts should match. A mismatch is
a real gap worth chasing, not a rounding error.


In [ ]:
from common import read_jdbc

for cfg in configs:
    source_rows = read_jdbc(spark, conn, cfg["source_table"]).count()
    target_rows = spark.table(cfg["target_table"]).count()
    flag = "ok" if source_rows == target_rows else "MISMATCH"
    print(f"{flag:9s} {cfg['source_table']:26s} source={source_rows:<8d} bronze={target_rows}")

### Idempotency check

Re-run cell 4. The row counts above should not change: the overlap window
re-reads a few rows, and the MERGE updates them in place instead of appending
duplicates. That is what makes a re-run safe.
